In [ ]:
set.seed(1)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(tidyverse)
library(patchwork)
library(harmony)
library(readxl)
library(viridis)

In [ ]:
date <- "2026-06-05_online_develop_all"

In [ ]:
data_dir <- "/project/work/Data/"

# 准备数据

In [ ]:
obj <- readRDS("/project/resources/2026-05-06_rna_all_sample/Data/2026-05-07_filtered_developing_by_donor.rds")

In [ ]:
new_df <- readRDS("/project/resources/2026-05-06_rna_all_sample/Data/2026-05-13_human_rna_gene_name.rds")
current_genes <- rownames(obj)

match_idx <- match(current_genes, new_df$GeneName)
new_gene_names <- new_df$hgnc_symbol[match_idx]

is_missing <- is.na(new_gene_names) | new_gene_names == ""
new_gene_names[is_missing] <- current_genes[is_missing]

new_gene_names <- make.unique(new_gene_names)

In [ ]:
counts_matrix <- GetAssayData(obj, assay = "RNA", layer = "counts")
head(counts_matrix)

In [ ]:
rownames(counts_matrix) <- new_gene_names

In [ ]:
obj[["RNA"]] <- CreateAssayObject(counts = counts_matrix)

In [ ]:
counts_matrix <- GetAssayData(obj, assay = "RNA", layer = "counts")
head(counts_matrix)

In [ ]:
head(rownames(counts_matrix),50)
tail(rownames(counts_matrix),50)

# QC

In [ ]:
obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern = "^MT-")

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 5)
VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA","percent.mt"), ncol = 4, group.by = "donor_id", pt.size = 0)

In [ ]:
dim(obj)
qs <- quantile(obj$nFeature_RNA, probs = c(0.05, 0.95), na.rm = TRUE)
qs
# obj <- subset(obj,subset = nFeature_RNA >= qs[[1]] & nFeature_RNA <= qs[[2]])
dim(obj)

# Process analysis

In [ ]:
obj[["RNA"]] <- split(obj[["RNA"]], f = obj$donor_id)

In [ ]:
obj <- NormalizeData(obj, normalization.method = "LogNormalize", scale.factor = 10000)

In [ ]:
obj <- FindVariableFeatures(obj, selection.method = "vst", nfeatures = 3000)

In [ ]:
obj <- ScaleData(obj, features = VariableFeatures(obj))

In [ ]:
obj <- RunPCA(obj, features = VariableFeatures(obj), npcs = 40)
obj <- FindNeighbors(obj, dims = 1:20, reduction = "pca")

In [ ]:
obj <- FindClusters(obj, resolution = 0.4, cluster.name = "unintegrated_clusters04")
obj <- RunUMAP(obj, dims = 1:20, reduction = "pca", reduction.name = "umap.unintegrated")
# DimPlot(obj, reduction = "umap.unintegrated")

In [ ]:
saveRDS(obj,file = paste0(data_dir, date, "_human_unintegrated.rds")) # 已经删除

In [ ]:
paste0(data_dir, date, "_human_unintegrated.rds")

# harmony

In [ ]:
obj <- IntegrateLayers(object = obj, method = HarmonyIntegration, orig.reduction = "pca", new.reduction = "harmony", verbose = FALSE)

In [ ]:
obj[["RNA"]] <- JoinLayers(obj[["RNA"]])

In [ ]:
obj <- FindNeighbors(obj, reduction = "harmony", dims = 1:20)
obj <- FindClusters(obj, resolution = 0.4, cluster.name = "harmony_clusters04")

In [ ]:
obj <- RunUMAP(obj, reduction = "harmony", dims = 1:20, reduction.name = "umap.harmony")

In [ ]:
saveRDS(obj,file = paste0(data_dir, date, "_human_harmony.rds"))

In [ ]:
paste0(data_dir, date, "_human_harmony.rds")

# read

In [ ]:
obj <- readRDS("/project/work/Data/2026-06-05_online_develop_all_human_harmony.rds")
obj

In [ ]:
obj$donor_age <- obj$development_stage

In [ ]:
obj$tissue_age <- paste0(obj$tissue, "_", obj$donor_age)

In [ ]:
options(repr.plot.width = 13, repr.plot.height =8)
DimPlot(obj, reduction = "umap.unintegrated", group.by = "tissue_age",raster=FALSE)

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
DimPlot(obj, reduction = "umap.unintegrated", group.by = "majorclass",raster=FALSE)

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
DimPlot(obj, reduction = "umap.unintegrated", group.by = "unintegrated_clusters04",raster=FALSE)

In [ ]:
options(repr.plot.width = 13, repr.plot.height =8)
DimPlot(obj, reduction = "umap.harmony", group.by = "tissue_age",raster=FALSE)

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
DimPlot(obj, reduction = "umap.harmony", group.by = "majorclass", label= TRUE,raster=FALSE)

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
DimPlot(obj, reduction = "umap.harmony", group.by = "harmony_clusters04",raster=FALSE)

In [ ]:
options(repr.plot.width = 24, repr.plot.height = 24)
DimPlot(obj, reduction = "umap.harmony", split.by = "tissue_age",raster=FALSE,ncol=4)

In [ ]:
CONTACT_EMAIL <- CONTACT_EMAIL[,c('orig.ident','nCount_RNA','nFeature_RNA','percent.mt','assay','disease',
                                  'donor_id','donor_age','sex','self_reported_ethnicity','tissue',
                                  'cell_type','majorclass',#'subclass',
                                  # 'doublet_info',
                                  'unintegrated_clusters04','seurat_clusters','harmony_clusters04')]

In [ ]:
obj$tissue <- as.character(obj$tissue)

In [ ]:
obj$tissue[obj$tissue == 'macula lutea'] = 'macula'
obj$tissue[obj$tissue == 'peripheral region of retina'] = 'peripheral'

In [ ]:
obj$donor_age <- as.character(obj$donor_age)

In [ ]:
obj$donor_age[obj$donor_id == 'Donor_1'] = '70days'
obj$donor_age[obj$donor_id == 'Donor_2'] = '79days'
obj$donor_age[obj$donor_id == 'Donor_3'] = '87days'
obj$donor_age[obj$donor_id == 'Donor_4'] = '91days'
obj$donor_age[obj$donor_id == 'Donor_5'] = '100days'
obj$donor_age[obj$donor_id == 'Donor_6'] = '103days'
obj$donor_age[obj$donor_id == 'Donor_7'] = '116days'
obj$donor_age[obj$donor_id == 'Donor_8'] = '137days'
obj$donor_age[obj$donor_id == 'Donor_9'] = '141days'
obj$donor_age[obj$donor_id == 'Donor_10'] = '142days'
obj$donor_age[obj$donor_id == 'Donor_11'] = '162days'
obj$donor_age[obj$donor_id == 'Donor_12'] = '165days'

In [ ]:
obj$tissue_age <- paste0(obj$tissue, "_", obj$donor_age)

In [ ]:
options(repr.plot.width = 10, repr.plot.height =8)
DimPlot(obj, reduction = "umap.harmony", group.by = "tissue_age",raster=FALSE)

In [ ]:
options(repr.plot.width = 24, repr.plot.height = 24)
DimPlot(obj, reduction = "umap.harmony", split.by = "tissue_age",raster=FALSE,ncol=4)

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 5)
VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA","percent.mt"), ncol = 3, group.by = "tissue_age", pt.size = 0)

# anno

In [ ]:
use.markers <- c(
  "SOX2", "PAX6", "RAX", "HES1", "MKI67",       # RPC
  "RHO", "PDE6A", "CNGA1", "SAG", "GNAT1",               # Rod
  "OPN1MW", "OPN1SW", "PDE6H", "ARR3", "GUCA1C",          # Cone
  "VSX1", "GRM6", "TRPM1", "PRKCA", "CABP5",             # BC
  "GAD1", "SLC6A9", "TFAP2A", "GAD2", "TFAP2B",          # AC
  "ONECUT1", "ONECUT2", "LHX1", "CALB1", "PROX1",        # HC
  "POU4F1", "POU4F2", "NEFL", "RBPMS", "ATOH7", "SNCG",  # RGC
  "RLBP1", "GLUL", "DKK3", "APOE", "SOX9"                 # MG
)

length(unique(use.markers))

In [ ]:
use.types <- c(
  # "NRPC", "NRPC", "NRPC", "NRPC", "NRPC",
  # "PRPC", "PRPC", "PRPC", "PRPC",
  "RPC", "RPC", "RPC", "RPC", "RPC", #"RPC",
  "Rod", "Rod", "Rod", "Rod", "Rod",
  "Cone", "Cone", "Cone", "Cone", "Cone",
  "BC", "BC", "BC", "BC", "BC",
  "AC", "AC", "AC", "AC", "AC",
  "HC", "HC", "HC", "HC", "HC",
  "RGC", "RGC", "RGC", "RGC", "RGC", "RGC",
  "MG", "MG", "MG", "MG", "MG"
)

length(use.types)

In [ ]:
use.markers.df <- data.frame(features.plot = use.markers, type = use.types)

## 注释harmony_clusters04

In [ ]:
p <- DotPlot(obj, features = use.markers, group.by = "harmony_clusters04",cluster.idents = TRUE)+
scale_colour_gradient2(low = "black", mid = "grey", high = "red")+ 
theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black"))

# p

In [ ]:
use.data <- p$data

In [ ]:
use.data <- merge(use.data, use.markers.df, by = "features.plot")

In [ ]:
use.data$features.plot <- factor(use.data$features.plot, levels = use.markers)

In [ ]:
use.data$type <- factor(use.data$type, levels = unique(use.types))

In [ ]:
p1 <- ggplot(use.data, aes(x = features.plot, y = id)) +
  geom_point(aes(size = pct.exp, color = avg.exp.scaled)) +  # 使用点大小和颜色表示信息
  scale_size(range = c(0.1, 5)) +  # 调整点的大小范围
  scale_color_gradient2(low = "black", mid = "grey", high = "red") +  # 定义颜色梯度
  facet_wrap(~ type, scales = "free_x", nrow = 1) +  # 根据类型分面
  theme_classic() +
  theme(text = element_text(color = "black"),
    axis.text.y = element_text(color = "black", size = 12),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black", size = 10),
    legend.title = element_text(size = 10),  # 图例标题字号及下方间距
    strip.text = element_text(size = 11)  # 调整分面标签字体大小
  ) +
  labs(x = "Features", y = "Cell Type", size = "Percentag\n Expressed", color = "Scaled\nExpression")

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
p1

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 5)
VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA","percent.mt"), ncol = 3, group.by = "harmony_clusters04", pt.size = 0)

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
DimPlot(obj, reduction = "umap.harmony", group.by = "harmony_clusters04", label = TRUE,raster=FALSE)

In [ ]:
obj$celltype <- paste0('UN_',obj$harmony_clusters04)

In [ ]:
obj$celltype[obj$harmony_clusters04 %in% c('1','9')] = 'Rod'
obj$celltype[obj$harmony_clusters04 %in% c('11')] = 'Cone'
obj$celltype[obj$harmony_clusters04 %in% c('')] = 'BC'
obj$celltype[obj$harmony_clusters04 %in% c('')] = 'AC'
obj$celltype[obj$harmony_clusters04 %in% c('')] = 'HC'
obj$celltype[obj$harmony_clusters04 %in% c('4','2','10','18')] = 'RGC'
obj$celltype[obj$harmony_clusters04 %in% c('')] = 'MG'
obj$celltype[obj$harmony_clusters04 %in% c('')] = 'RPC'
# obj$celltype[obj$harmony_clusters04 %in% c('')] = ''

## FindClusters : resolution = 1

In [ ]:
obj <- FindClusters(obj, resolution = 1, cluster.name = "harmony_clusters1")

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
DimPlot(obj, reduction = "umap.harmony", group.by = "harmony_clusters1", label = TRUE,raster=FALSE)

In [ ]:
p <- DotPlot(obj, features = use.markers, group.by = "harmony_clusters1",cluster.idents = TRUE)+
scale_colour_gradient2(low = "black", mid = "grey", high = "red")+ 
theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black"))

# p

In [ ]:
use.data <- p$data

In [ ]:
use.data <- merge(use.data, use.markers.df, by = "features.plot")

In [ ]:
use.data$features.plot <- factor(use.data$features.plot, levels = use.markers)

In [ ]:
use.data$type <- factor(use.data$type, levels = unique(use.types))

In [ ]:
p1 <- ggplot(use.data, aes(x = features.plot, y = id)) +
  geom_point(aes(size = pct.exp, color = avg.exp.scaled)) +  # 使用点大小和颜色表示信息
  scale_size(range = c(0.1, 5)) +  # 调整点的大小范围
  scale_color_gradient2(low = "black", mid = "grey", high = "red") +  # 定义颜色梯度
  facet_wrap(~ type, scales = "free_x", nrow = 1) +  # 根据类型分面
  theme_classic() +
  theme(text = element_text(color = "black"),
    axis.text.y = element_text(color = "black", size = 12),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black", size = 10),
    legend.title = element_text(size = 10),  # 图例标题字号及下方间距
    strip.text = element_text(size = 11)  # 调整分面标签字体大小
  ) +
  labs(x = "Features", y = "Cell Type", size = "Percentag\n Expressed", color = "Scaled\nExpression")

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 12)
p1

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 5)
VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA","percent.mt"), ncol = 3, group.by = "harmony_clusters1", pt.size = 0)

In [ ]:
obj$celltype <- paste0('UN_',obj$harmony_clusters1)

In [ ]:
obj$celltype[obj$harmony_clusters1 %in% c('22','25','2','0')] = 'Rod'
obj$celltype[obj$harmony_clusters1 %in% c('11','29')] = 'Cone'
obj$celltype[obj$harmony_clusters1 %in% c('41','37','20','16','15','13')] = 'BC'
obj$celltype[obj$harmony_clusters1 %in% c('23','6','39','12','27','8')] = 'AC'
obj$celltype[obj$harmony_clusters1 %in% c('32','9')] = 'HC'
obj$celltype[obj$harmony_clusters1 %in% c('40','21','35','19','7','30','3','17','38')] = 'RGC'
obj$celltype[obj$harmony_clusters1 %in% c('26','18','42')] = 'MG'
obj$celltype[obj$harmony_clusters1 %in% c('33','24','5','1','4','31','28','10','34','36','14')] = 'RPC'
# obj$celltype[obj$harmony_clusters1 %in% c('')] = ''

In [ ]:
options(repr.plot.width = 8, repr.plot.height =8)
DimPlot(obj, reduction = "umap.harmony", group.by = "celltype", label = TRUE,raster=FALSE)

In [ ]:
p <- DotPlot(obj, features = use.markers, group.by = "celltype",cluster.idents = TRUE)+
scale_colour_gradient2(low = "black", mid = "grey", high = "red")+ 
theme(axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black"))

# p

In [ ]:
use.data <- p$data

In [ ]:
use.data <- merge(use.data, use.markers.df, by = "features.plot")

In [ ]:
use.data$features.plot <- factor(use.data$features.plot, levels = use.markers)

In [ ]:
use.data$type <- factor(use.data$type, levels = unique(use.types))

In [ ]:
p1 <- ggplot(use.data, aes(x = features.plot, y = id)) +
  geom_point(aes(size = pct.exp, color = avg.exp.scaled)) +  # 使用点大小和颜色表示信息
  scale_size(range = c(0.1, 5)) +  # 调整点的大小范围
  scale_color_gradient2(low = "black", mid = "grey", high = "red") +  # 定义颜色梯度
  facet_wrap(~ type, scales = "free_x", nrow = 1) +  # 根据类型分面
  theme_classic() +
  theme(text = element_text(color = "black"),
    axis.text.y = element_text(color = "black", size = 12),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black", size = 10),
    legend.title = element_text(size = 10),  # 图例标题字号及下方间距
    strip.text = element_text(size = 11)  # 调整分面标签字体大小
  ) +
  labs(x = "Features", y = "Cell Type", size = "Percentag\n Expressed", color = "Scaled\nExpression")

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 8)
p1

# save

In [ ]:
obj$donor_age <- factor(obj$donor_age,level = c('70days','79days','87days','91days','100days','103days','116days','137days','141days','142days','162days','165days'))
obj$celltype <- factor(obj$celltype,level = c('Rod','Cone','AC','BC','HC','MG','RGC','RPC'))
obj$tissue <- factor(obj$tissue,level = c('macula','peripheral'))

In [ ]:
obj$tissue_age <- factor(obj$tissue_age,level = c('macula_70days','peripheral_70days','macula_79days','peripheral_79days','macula_87days','peripheral_87days',
                                                  'macula_91days','peripheral_91days','macula_100days','peripheral_100days','macula_103days','peripheral_103days',
                                                  'macula_116days','peripheral_116days','macula_137days','peripheral_137days','macula_141days','peripheral_141days','macula_142days',
                                                  'peripheral_142days','macula_162days','peripheral_162days','macula_165days','peripheral_165days'))

In [ ]:
saveRDS(obj,"/project/work/Data/2026-06-05_develop_all_human_harmony_anno.rds") # 最后更名为'2026-06-05_develop_all_human_harmony_anno.rds'